In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

단어 사전 구축 및 토큰화

In [30]:
vocab = {
    "<PAD>": 0,
    "<SOS>": 1,
    "<EOS>": 2,
    "I": 3,
    "am": 4,
    "a": 5,
    "student": 6,
    "teacher": 7,
}

# ID를 다시 단어로 바꿀 역사전
idx2word = {idx: word for word, idx in vocab.items()}

In [ ]:
# 입력 문장 토큰화: "I am a student" -> [3, 4, 5, 6]
input_text = ["I", "am", "a", "student"]
decode_text = ["나", "는", "학생이다."]
input_indices = torch.tensor([[vocab[w] for w in input_text]]) # Shape: [1, 4] (Batch=1, Seq_Len=4)
print("input_indices shape: ", input_indices.shape)
input_indices

input_indices shape:  torch.Size([1, 4])


tensor([[3, 4, 5, 6]])

- 인코더 레이어 정의

In [119]:
class AttentionEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.hidden_dim = hidden_dim
        self.rnn = nn.LSTM(self.embedding_dim, self.hidden_dim, batch_first=True)

    def forward(self, src):
        embedding = self.embedding(src)
        output, (hidden, cell) = self.rnn(embedding)

        return output, (hidden, cell)


- 어텐션 레이어 정의

In [56]:
class LuongAttention(nn.Module):
    def forward(self, decoder_hidden, encoder_out):
        # 현재 디코더의 은닉상태 s_t 를 차원 확장 
        # decoder_hidden Shape: [Batch, Hidden_Dim]
        Q = decoder_hidden.unsqueeze(2)
        K = encoder_out
        V = encoder_out
        # Q Shape: [Batch, Hidden_Dim, 1]
        # K Shape: [Batch, Seq_Len, Hidden_Dim]

        # 인코더에서 넘어오는 은닉 상태 행렬과 내적 K * Q
        attention_scores = torch.bmm(K, Q) 
        # attention_scores Shape: [Batch, Seq_Len, 1]

        # 어텐션 weight 계산 (Softmax)
        attention_scores = attention_scores.squeeze(2)
        # attention_scores Shape : [Batch, Seq_Len]
        attention_weights = F.softmax(attention_scores, dim=1)
      
        # 어텐션 값 계산 (attention_weight x V)
        attention_weights = attention_weights.unsqueeze(1)
        # attention_weights Shape: [Batch, Seq_Len] -> [Batch, 1, Seq_Len]

        context = torch.bmm(attention_weights, V)
        # context Shape: [Batch, 1, Hidden_Dim]

        context = context.squeeze(1)
        # [Batch, 1 , Hidden_Dim] -> [Batch, Hidden_Dim]

        return context, attention_weights


 - 디코더 레이어 정의

In [165]:
class SimpleAttentionDecoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.attention = LuongAttention()
        self.concat = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward_step(self, input_token, pre_hidden, pre_cell, encoder_outputs):
        embedded = self.embedding(input_token)
        if embedded.dim() == 2:
            embedded = embedded.unsqueeze(1)  # [B, 1, E]
            
        current_output, (current_hidden, current_cell) = self.rnn(embedded, (pre_hidden, pre_cell))
        s_t = current_hidden.squeeze(0) # [1, B, H] -> [B, H]
        # 어텐션 값 (Context Vector) 추출
        context, attn_weights = self.attention(s_t, encoder_outputs)

        # 디코더 상태 + 어텐션 값 결합
        concat_input = torch.cat((s_t, context), dim=1)
        concat_output = torch.tanh(self.concat(concat_input))

        # 최종 예측
        output_logits = self.fc_out(concat_output)
        return output_logits, current_hidden, current_cell, attn_weights

- 실행 래퍼

In [166]:
class AttentionSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        encoder_out, (hidden, cell) = self.encoder(src)
        outputs = []
        input_token = trg[:, 0]  # <SOS>
        for t in range(1, trg.shape[1]):
            logits, hidden, cell, _ = self.decoder.forward_step(
                input_token, hidden, cell, encoder_out
            )
            outputs.append(logits)
            input_token = trg[:, t]  # Teacher Forcing
        return torch.stack(outputs, dim=1)  # [B, T-1, V]

- 유틸리티 함수

In [167]:
# 단어 리스트를 vocab ID 텐서로 바꾼다.
def encode(tokens, vocab, add_eos=True):
    ids = [vocab[t] for t in tokens]
    if add_eos:
        ids.append(vocab["<EOS>"])
    return torch.tensor(ids)


In [168]:
# 문장 여러개를 같은 모양의 텐서 배치로 묶는 함수
# str일 경우 길이가 달라 텐서로 묶을 수 없기 때문에 단어 vocab의 idx 값으로 치환한다.
# 이때 문장의 길이가 각각 다르기 때문에 문장의 길이가 짧은 쪽을 <PAD>를 추가해서 매치시킨다.
def collate(pairs, src_vocab, trg_vocab):
    srcs, trgs = [], []
    for src_toks, trg_toks in pairs:
        # src, trgs 각각 vocab의 idx에 매치해서 단일 행 텐서로 변환 하는 작업
        # src = tensor( [[3,4,5,6,2], # 첫 번째 문장
        #                [3,4,5,7,2]])# 두 번째 문장
        srcs.append(encode(src_toks, src_vocab, add_eos=True))
        trgs.append(torch.tensor(
            [trg_vocab["<SOS>"]] + [trg_vocab[t] for t in trg_toks] + [trg_vocab["<EOS>"]]
        ))

    # PAD 하기 전 각각 배치에서 가장 긴 문장 길이를 찾는 항
    max_s = max(s.size(0) for s in srcs)
    max_t = max(t.size(0) for t in trgs)

    src_batch = torch.stack([F.pad(s, (0, max_s - s.size(0))) for s in srcs])
    trg_batch = torch.stack([F.pad(t, (0, max_t - t.size(0))) for t in trgs])
    return src_batch, trg_batch

- 코드 진행

In [172]:
src_vocab = {"<PAD>": 0,"<SOS>": 1,"<EOS>": 2,"I": 3,"am": 4,"a": 5,"student": 6,"teacher": 7}
trg_vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "나는": 3, "학생": 4, "이다": 5, "선생님": 6}
pairs = [
    (["I", "am", "a", "student"], ["나는", "학생", "이다"]),
    (["I", "am", "a", "teacher"], ["나는", "선생님", "이다"]),
]

# collate로 src_batch, trg_batch 제작
src_batch, trg_batch = collate(pairs, src_vocab, trg_vocab)

encoder = AttentionEncoder(len(src_vocab), 32, 64)
decoder = SimpleAttentionDecoder(len(trg_vocab), 32, 64)  # vocab_size만 타겟으로
model = AttentionSeq2Seq(encoder, decoder)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss(ignore_index=trg_vocab["<PAD>"])

model.train()
for epoch in range(300):
    if (epoch + 1) % 50 == 0:
        print(f"Epoch: {epoch+1} Loss: {loss.item()}")
    optimizer.zero_grad()
    logits = model(src_batch, trg_batch)
    loss = criterion(logits.reshape(-1, logits.size(-1)), trg_batch[:, 1:].reshape(-1))
    loss.backward()
    optimizer.step()

Epoch: 50 Loss: 0.00012706415145657957
Epoch: 100 Loss: 8.464835264021531e-05
Epoch: 150 Loss: 6.863159069325775e-05
Epoch: 200 Loss: 5.724816583096981e-05
Epoch: 250 Loss: 4.862103014602326e-05
Epoch: 300 Loss: 4.191590778646059e-05


In [173]:
@torch.no_grad()
def translate(tokens, max_len=10):
    model.eval()
    src = encode(tokens, src_vocab, add_eos=True).unsqueeze(0)  # [1, S]
    encoder_outputs, hidden, cell = model.encoder(src)

    input_token = torch.tensor([trg_vocab["<SOS>"]])
    generated = []
    for _ in range(max_len):
        logits, hidden, cell, weights = model.decoder.forward_step(
            input_token, hidden, cell, encoder_outputs
        )
        pred = logits.argmax(dim=1)
        word = trg_idx2word[pred.item()]
        if word == "<EOS>":
            break
        if word != "<PAD>":
            generated.append(word)
        input_token = pred
    return " ".join(generated)


print("\n=== 학습 후 번역 결과 ===")
print("I am a student  ->", translate(["I", "am", "a", "student"]))
print("I am a teacher  ->", translate(["I", "am", "a", "teacher"]))


=== 학습 후 번역 결과 ===


ValueError: not enough values to unpack (expected 3, got 2)

In [78]:
# ==========================================
# 수정판: 학습 가능한 Attention Seq2Seq (영→한 토이 예제)
# - Embedding을 __init__에 등록
# - LSTM batch_first / hidden shape 정리
# - 타겟 vocab + 병렬 데이터 + CrossEntropy 학습
# ==========================================
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

# 1) 소스/타겟 단어 사전
src_vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "I": 3, "am": 4, "a": 5, "student": 6, "teacher": 7}
trg_vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "나는": 3, "학생": 4, "이다": 5, "선생님": 6}
src_idx2word = {i: w for w, i in src_vocab.items()}
trg_idx2word = {i: w for w, i in trg_vocab.items()}

# 토이 병렬 코퍼스
pairs = [
    (["I", "am", "a", "student"], ["나는", "학생", "이다"]),
    (["I", "am", "a", "teacher"], ["나는", "선생님", "이다"]),
]

def encode(tokens, vocab, add_eos=True):
    ids = [vocab[t] for t in tokens]
    if add_eos:
        ids.append(vocab["<EOS>"])
    return torch.tensor(ids)

# 2) 모델 정의 (버그 수정)
class FixedEncoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)  # __init__에 등록!
        self.rnn = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        # src: [B, S]
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.rnn(embedded)
        return outputs, hidden, cell  # outputs: [B, S, H]


class FixedLuongAttention(nn.Module):
    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden: [B, H], encoder_outputs: [B, S, H]
        query = decoder_hidden.unsqueeze(2)              # [B, H, 1]
        scores = torch.bmm(encoder_outputs, query)       # [B, S, 1]
        weights = F.softmax(scores.squeeze(2), dim=1)    # [B, S]
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)  # [B, H]
        return context, weights


class FixedDecoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.attention = FixedLuongAttention()
        self.concat = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward_step(self, input_token, hidden, cell, encoder_outputs):
        # input_token: [B], hidden/cell: [1, B, H]
        embedded = self.embedding(input_token).unsqueeze(1)  # [B, 1, E]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))

        # 현재 디코더 상태 s_t 사용
        s_t = hidden.squeeze(0)  # [B, H]
        context, attn_weights = self.attention(s_t, encoder_outputs)

        concat_out = torch.tanh(self.concat(torch.cat([s_t, context], dim=1)))
        logits = self.fc_out(concat_out)  # [B, vocab]
        return logits, hidden, cell, attn_weights


class AttentionSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        # Teacher Forcing 학습: trg = [<SOS>, w1, w2, ..., <EOS>]
        encoder_outputs, hidden, cell = self.encoder(src)
        batch_size, trg_len = trg.shape
        outputs = []

        input_token = trg[:, 0]  # <SOS>
        for t in range(1, trg_len):
            logits, hidden, cell, _ = self.decoder.forward_step(
                input_token, hidden, cell, encoder_outputs
            )
            outputs.append(logits)
            input_token = trg[:, t]  # 다음 정답 토큰
        return torch.stack(outputs, dim=1)  # [B, trg_len-1, vocab]


# 3) 학습
emb_dim, hidden_dim = 32, 64
encoder = FixedEncoder(len(src_vocab), emb_dim, hidden_dim)
decoder = FixedDecoder(len(trg_vocab), emb_dim, hidden_dim)
model = AttentionSeq2Seq(encoder, decoder)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss(ignore_index=trg_vocab["<PAD>"])

# 배치 텐서 만들기 (패딩)
def collate(pairs):
    srcs, trgs = [], []
    for src_toks, trg_toks in pairs:
        srcs.append(encode(src_toks, src_vocab, add_eos=True))
        # 디코더 입력용: <SOS> + 타겟 + <EOS>
        trgs.append(torch.tensor(
            [trg_vocab["<SOS>"]] + [trg_vocab[t] for t in trg_toks] + [trg_vocab["<EOS>"]]
        ))
    max_s = max(s.size(0) for s in srcs)
    max_t = max(t.size(0) for t in trgs)
    src_batch = torch.stack([F.pad(s, (0, max_s - s.size(0))) for s in srcs])
    trg_batch = torch.stack([F.pad(t, (0, max_t - t.size(0))) for t in trgs])
    return src_batch, trg_batch

src_batch, trg_batch = collate(pairs)

model.train()
for epoch in range(300):
    optimizer.zero_grad()
    logits = model(src_batch, trg_batch)          # [B, T-1, V]
    loss = criterion(logits.reshape(-1, logits.size(-1)), trg_batch[:, 1:].reshape(-1))
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | loss = {loss.item():.4f}")


# 4) 추론 (그리디 디코딩)
@torch.no_grad()
def translate(tokens, max_len=10):
    model.eval()
    src = encode(tokens, src_vocab, add_eos=True).unsqueeze(0)  # [1, S]
    encoder_outputs, hidden, cell = model.encoder(src)

    input_token = torch.tensor([trg_vocab["<SOS>"]])
    generated = []
    for _ in range(max_len):
        logits, hidden, cell, weights = model.decoder.forward_step(
            input_token, hidden, cell, encoder_outputs
        )
        pred = logits.argmax(dim=1)
        word = trg_idx2word[pred.item()]
        if word == "<EOS>":
            break
        if word != "<PAD>":
            generated.append(word)
        input_token = pred
    return " ".join(generated)


print("\n=== 학습 후 번역 결과 ===")
print("I am a student  ->", translate(["I", "am", "a", "student"]))
print("I am a teacher  ->", translate(["I", "am", "a", "teacher"]))


Epoch  50 | loss = 0.0002
Epoch 100 | loss = 0.0001
Epoch 150 | loss = 0.0001
Epoch 200 | loss = 0.0001
Epoch 250 | loss = 0.0001
Epoch 300 | loss = 0.0000

=== 학습 후 번역 결과 ===
I am a student  -> 나는 학생 이다
I am a teacher  -> 나는 선생님 이다


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# 1. 단어 사전(Vocabulary) 구축 및 토큰화
# ==========================================
vocab = {
    "<PAD>": 0, "<SOS>": 1, "<EOS>": 2,
    "I": 3, "am": 4, "a": 5, "student": 6, "teacher": 7
}
# ID를 다시 단어로 바꿀 역사전
idx2word = {idx: word for word, idx in vocab.items()}

# 입력 문장 토큰화: "I am a student" -> [3, 4, 5, 6]
input_text = ["I", "am", "a", "student"]
input_indices = torch.tensor([[vocab[w] for w in input_text]]) # Shape: [1, 4] (Batch=1, Seq_Len=4)


# ==========================================
# 2. 어텐션 레이어 및 디코더 정의
# ==========================================
class LuongAttention(nn.Module):
    def forward(self, decoder_hidden, encoder_outputs):
        # Q: [Batch, Hidden_Dim, 1]
        query = decoder_hidden.unsqueeze(2)
        
        # 1. 어텐션 스코어 (Q · K): [Batch, Seq_Len, 1]
        scores = torch.bmm(encoder_outputs, query)
        
        # 2. 어텐션 가중치 (Softmax): [Batch, Seq_Len]
        weights = F.softmax(scores.squeeze(2), dim=1)
        
        # 3. 어텐션 값 (Score x V): [Batch, Hidden_Dim]
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
        
        return context, weights

class SimpleAttentionDecoder(nn.Module):
    def __init__(self, vocab_size, hidden_dim):
        super().__init__()
        self.attention = LuongAttention()
        self.concat = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, decoder_hidden, encoder_outputs):
        # 어텐션 값(Context Vector) 추출
        context, attn_weights = self.attention(decoder_hidden, encoder_outputs)
        
        # 디코더 상태 + 어텐션 값 결합(Concat)
        concat_input = torch.cat((decoder_hidden, context), dim=1)
        concat_output = torch.tanh(self.concat(concat_input))
        
        # 최종 단어 예측
        output_logits = self.fc_out(concat_output)
        return output_logits, attn_weights


# ==========================================
# 3. 모델 파라미터 초기화 및 데이터 실행
# ==========================================
torch.manual_seed(42) # 결과 재현을 위한 시드 고정

embedding_dim = 16
hidden_dim = 16
vocab_size = len(vocab)

# (1) 임베딩 레이어 & 인코더(RNN) 생성
embedding = nn.Embedding(vocab_size, embedding_dim)
encoder_rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)

# (2) 입력 단어들을 벡터로 변환 후 인코더 RNN 통과
embedded_inputs = embedding(input_indices) # [1, 4, 16]
encoder_outputs, last_hidden = encoder_rnn(embedded_inputs) # K, V 역할인 h_n 생성!

# (3) 디코더 및 어텐션 실행
decoder = SimpleAttentionDecoder(vocab_size, hidden_dim)

# 디코더의 초기 상태(Q)는 인코더의 마지막 상태를 사용해보겠습니다.
decoder_hidden = last_hidden.squeeze(0) 

# 실행!
logits, attn_weights = decoder(decoder_hidden, encoder_outputs)


# ==========================================
# 4. 결과 출력
# ==========================================
print("=== 입력 문장 ===")
print(" -> ".join(input_text))
print("\n=== 각 단어(h_n)에 부여된 어텐션 가중치 비율 ===")
for word, weight in zip(input_text, attn_weights[0].tolist()):
    print(f"단어 '{word:7s}' 에 대한 가중치: {weight * 100:.2f}%")

# 가장 예측 확률이 높은 단어 출력
predicted_idx = torch.argmax(logits, dim=1).item()
print(f"\n=== 디코더가 최종 예측한 단어 ===")
print(f"예측된 단어: '{idx2word[predicted_idx]}'")